In [5]:

%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

matplotlib.use('Qt5Agg')  # Asegúrate de que este backend está instalado.
mne.viz.set_browser_backend('qt')  # Activa el backend interactivo

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

from neurodsp.spectral import compute_spectrum, trim_spectrum
from neurodsp.plts import plot_power_spectra

# Import IRASA related functions
from neurodsp.aperiodic import compute_irasa, fit_irasa

from joblib import Parallel, delayed

import pickle

In [6]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_SELF import get_paths_SELF

# Parámetros editables
disco = "g"
layer_script = "event"
subj = "s01b"


# Generar variables automáticamente
path_dict = get_paths_SELF(disco=disco, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")



✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\ICA_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_matlab_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\evoked_event
✅ Carpeta creada: g:\PROYECTO_SELF\channels_structure
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\raw_hsp
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\fwd
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\inverse
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\acw_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\PLE_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\ISC_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\anal

In [7]:
f_min=0.5
f_max=40


pickle_file = epochs_clean_path / f"dict_conditions.pkl"
# Cargar el pickle
with open(pickle_file, "rb") as f:
    dict_conditions = pickle.load(f)

#epochs
combinaciones = list(dict_conditions.keys())
print(f"combinaciones: {combinaciones}")

# print(subj)
subjects = sorted({f.name.split("_")[0].lower() for f in data_task_edf.glob("*.edf")})
print(f"subj: {subjects}")


## IS THIS NECCESSARY?
# channels = pd.read_csv(channels_structure_path / f"channels_mag_{modality}.csv")
# channels_mag=channels[channels[f"canal_efectivo_{modality}"].notna()][f"canal_efectivo_{modality}"]
# channels_mag=channels_mag.tolist()
# del channels



edf_file = data_task_edf / f"{subjects[0]}_vis_c_BVica-export.edf"
elp_file = data_task_edf / f"{subjects[0]}_vis_c_BVica-export.elp"

# -------------------------
# 2. Leer EDF
# -------------------------

# Leer el EDF
raw = mne.io.read_raw_edf(edf_file, preload=True)
channels_eeg = mne.pick_types(raw.info, eeg=True, meg=False, eog=False, exclude=[])
channels_eeg_names = [raw.ch_names[i] for i in channels_eeg]

combinaciones: ['self_pos', 'self_neu', 'self_neg', 'friend_pos', 'friend_neu', 'friend_neg', 'unk_pos', 'unk_neu', 'unk_neg']
subj: ['s01b', 's02b', 's03b', 's04b', 's05b', 's06b', 's07b', 's08b', 's09b', 's10b', 's11b', 's12b', 's13b', 's14b', 's15b', 's16b', 's17b', 's18b', 's19b', 's20b', 's21b', 's22b', 's23b', 's24b', 's25b', 's26b', 's27b', 's28b', 's29b']
Extracting EDF parameters from F:\WORKAREA\Datos SELF\Self_Exp2\Exp2_review\Visual_Corregidos\ICA\EDF\s01b_vis_c_BVica-export.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2465499  =      0.000 ...  4930.998 secs...


In [ ]:
## obtener power spechtum de el objeto evoked -- esto serñía el MIXED POWER 

# psds=evoked.compute_psd(method='welch', fmin=fmin, fmax=fmax, reject_by_annotation=True, n_fft = int(6 * sfreq))
# psds_all.append(psds)
            
# psds,freqs= mne.time_frequency.psd_array_welch(array, sfreq, fmin=0, fmax=inf, n_fft=256, n_overlap=0, 
# #                                    n_per_seg=None, n_jobs=None, average='mean', window='hamming', remove_dc=True, *, 
# #                                    output='power', verbose=None)



##please note THAT THIS IS ONLY FOR EVOKED, YOU NEED TO CALCULATE IT THROUGH EPOCHS

# ##def compute_psd(self, fmin=0, fmax=np.inf, tmin=None, tmax=None, proj=False)
# array=evoked.get_data()
# sfreq=evoked.info['sfreq']
# fmin=0.5
# fmax=40
# nfft=1024 # to increase the spectral resolution, , although you can go to 4096 if you want to see more details
# njobs=10

# ##compute psd FOR WHOLE SIGNAL
# psds,freqs= mne.time_frequency.psd_array_welch(array, sfreq, fmin=fmin, fmax=fmax, n_fft=nfft, n_jobs=njobs)



In [14]:
def compute_ple(subj, epochs, condition, f_range=(0.1, 40), hset=None, thresh=None, isplot=False, crop=None):
    
    if crop is not None:
        epochs.crop(tmin=crop)  # recorta desde 'crop' segundos en adelante
    # Obtener solo MEG sin canales marcados como bads
    data_epochs = epochs.copy().pick(picks="eeg", exclude="bads").get_data()
    channels_mag = epochs.copy().pick(picks="eeg", exclude="bads").ch_names

    sfreq = epochs.info['sfreq']
    duration = epochs.tmax - epochs.tmin

    # Contenedores globales
    psd_periodic_elect_all_epoch_all = []
    psd_aperiodic_elect_all_epoch_all = []
    slope_elect_all_epoch_all = []
    intercept_y_elect_all_epoch_all = []

    # Función paralelizable: IRASA + ajuste de pendiente
    def compute_irasa_fit(data_elect):
        freqs, psd_aperiodic, psd_periodic = compute_irasa(
            data_elect, fs=sfreq, f_range=f_range, hset=hset, thresh=thresh
        )
        intercept, slope = fit_irasa(freqs, psd_aperiodic)
        return psd_aperiodic, psd_periodic, intercept, slope, freqs

    # Procesar cada época
    for i, data_epoch in enumerate(data_epochs):
        # Paralelizar por canal
        results = Parallel(n_jobs=-1)(
            delayed(compute_irasa_fit)(data_epoch[ch])
            for ch in range(len(data_epoch))
        )

        # Extraer resultados
        psd_aperiodic_elect_all_epoch = [r[0] for r in results]
        psd_periodic_elect_all_epoch = [r[1] for r in results]
        intercept_y_elect_all_epoch = [r[2] for r in results]
        slope_elect_all_epoch = [r[3] for r in results]
        freqs = results[0][4]

        # Agregar por época
        psd_aperiodic_elect_all_epoch_all.append(psd_aperiodic_elect_all_epoch)
        psd_periodic_elect_all_epoch_all.append(psd_periodic_elect_all_epoch)
        intercept_y_elect_all_epoch_all.append(intercept_y_elect_all_epoch)
        slope_elect_all_epoch_all.append(slope_elect_all_epoch)

    # Construir DataFrame
    num_epochs = len(data_epochs)
    num_elects = len(channels_mag)
    shape_tabla = num_epochs * num_elects

    psd_periodic_elect_all_epoch_all_list = np.array(psd_periodic_elect_all_epoch_all).reshape(shape_tabla, -1).tolist()
    psd_aperiodic_elect_all_epoch_all_list = np.array(psd_aperiodic_elect_all_epoch_all).reshape(shape_tabla, -1).tolist()

    table_PLE = pd.DataFrame({
        'Subject': [subj] * shape_tabla,
        'Condition': [condition] * shape_tabla,
        'freqs': [freqs] * shape_tabla,
        'Epoch': np.repeat(np.arange(num_epochs), num_elects),
        'Elect': np.tile(channels_mag, num_epochs),
        'psd_periodic_elect_all_epoch_all': psd_periodic_elect_all_epoch_all_list,
        'psd_aperiodic_elect_all_epoch_all': psd_aperiodic_elect_all_epoch_all_list,
        'intercept_y_elect_all_epoch_all': np.array(intercept_y_elect_all_epoch_all).flatten(),
        'slope_elect_all_epoch_all': np.array(slope_elect_all_epoch_all).flatten()
    })

    return table_PLE



In [ ]:
# ## compute power law exponent for each epoch
# def ple_exponent(psds_all, isplot=False):
#     ##prueba con epochs, ahora has de adaptarlo a la función de power_spectrum_nans 
#     # first you need to obtain the frequencies, 
#     freqs= psds_all[0].freqs
#     #for np.polyfit we need to convert the frequencies to array
#     psds_array = [psds.get_data() for psds in psds_all]
#     psds_avg = [np.mean(epoch_psd, axis=0) for epoch_psd in psds_array]
#     psds_grand_avg = np.mean(psds_avg, axis=0)

#     coeffs= np.polyfit(np.log10(freqs), np.log10(psds_grand_avg), deg=1)
#     PLE= -coeffs[0]
#     if isplot==True:
#         #plot of the power spectrum
#         plt.loglog(freqs, psds_grand_avg,  label='PSD Mean')
#         #plot of the coeffs calculated
#         #its 10**y, where y is the linear regression, ax+b, a is slope, x is log10(freqs) and b is the intercept
#         y=coeffs[1] + coeffs[0]*np.log10(freqs)
#         plt.loglog(freqs, 10**y, 'r--', label=f'Fit PLE = {PLE:.2f}')
#         plt.xlabel('Frequency (Hz)')
#         plt.ylabel('Power Spectral Density (dB/Hz)')
#         plt.title('Power Law Exponent Fit using MNE-Python')
#         plt.legend()
#         plt.grid(True)
#         plt.show()

#         plt.show()



#     return coeffs, PLE, psds_grand_avg

In [15]:
##codigo para agrupar todas las tablas

all_tables = []

for i in range(0,len(subjects)):
    for h in range(0,len(combinaciones)):
        try:
            subj=subjects[i]
            combinacion= combinaciones[h]
            path_epochs= epochs_clean_path / f"{subj}_epochs_{combinacion}_{layer_script}-epo.fif"            
            epochs = mne.read_epochs(path_epochs)
            table_PLE=compute_ple(subj,epochs, condition=combinacion, f_range=(f_min, f_max),crop=0.5)
            all_tables.append(table_PLE)
            del epochs
        except Exception as e:
            print(f"Error en {subj} {combinacion}: {e}")
            continue

table_PLE_subjects_all = pd.concat(all_tables, ignore_index=True)

Reading g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s01b_epochs_self_pos_event-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    9000.00 ms
        0 CTF compensation matrices available
Not setting metadata
32 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_10524\949836760.py:6: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs = epochs.copy().pick(picks="eeg", exclude="bads").get_data()


Reading g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s01b_epochs_self_neu_event-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    9000.00 ms
        0 CTF compensation matrices available
Not setting metadata
32 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_10524\949836760.py:6: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs = epochs.copy().pick(picks="eeg", exclude="bads").get_data()


Reading g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s01b_epochs_self_neg_event-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    9000.00 ms
        0 CTF compensation matrices available
Not setting metadata
27 matching events found
No baseline correction applied
0 projection items activated


...

C:\Users\UCM\AppData\Local\Temp\ipykernel_10524\949836760.py:6: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs = epochs.copy().pick(picks="eeg", exclude="bads").get_data()


Reading g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s29b_epochs_unk_neu_event-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    9000.00 ms
        0 CTF compensation matrices available
Not setting metadata
24 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_10524\949836760.py:6: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs = epochs.copy().pick(picks="eeg", exclude="bads").get_data()


Reading g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s29b_epochs_unk_neg_event-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    9000.00 ms
        0 CTF compensation matrices available
Not setting metadata
30 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_10524\949836760.py:6: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs = epochs.copy().pick(picks="eeg", exclude="bads").get_data()


In [ ]:
table_PLE_subjects_all

In [16]:
table_PLE_subjects_all.to_pickle(PLE_path / f"table_PLE_subjects_all_{layer_script}.pickle")

In [17]:
table_PLE_slope_intercept_subjects_all= table_PLE_subjects_all[['Subject', 'Condition', 'freqs', 'Epoch', 'Elect', 'intercept_y_elect_all_epoch_all', 'slope_elect_all_epoch_all']]

In [18]:
table_PLE_slope_intercept_subjects_all.to_pickle(PLE_path / f"table_PLE_slope_intercept_subjects_all_{layer_script}.pickle")

In [ ]:
table_PLE_slope_intercept_subjects_all